# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: fetch latest code</h2>
            <span style="color:#f71;">I'm continually improving these labs, adding more examples and exercises.
            At the start of each week, it's worth checking you have the latest code.<br/>
            First do a <a href="https://chatgpt.com/share/6734e705-3270-8012-a074-421661af6ba9">git pull and merge your changes as needed</a>. Any problems? Try asking ChatGPT to clarify how to merge - or contact me!<br/><br/>
            After you've pulled the code, from the llm_engineering directory, in a Cursor Terminal, run:<br/>
            <code>uv sync</code><br/>
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            This lab uses FREE models: Ollama (local, no API key needed) and Groq (free tier API). Install Ollama from https://ollama.com or get a free Groq API key from https://console.groq.com
            </span>
        </td>
    </tr>
</table>

In [71]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [72]:
# Check for Groq API key (free tier available)
load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')

if groq_api_key:
    print(f"✓ Groq API Key found (begins {groq_api_key[:4]})")
    print("  Using Groq's FREE tier API")
else:
    print("✗ Groq API Key not set")
    print("  Get a free key at: https://console.groq.com")

# Check if Ollama is available
print("\nChecking Ollama (local, free):")
try:
    result = subprocess.run(['ollama', 'list'], capture_output=True, text=True, timeout=5)
    print("✓ Ollama is installed and available")
    print("\nAvailable models:")
    print(result.stdout)
except FileNotFoundError:
    print("✗ Ollama not found. Please install from https://ollama.com")
except Exception as e:
    print(f"Error checking Ollama: {e}")

✓ Groq API Key found (begins gsk_)
  Using Groq's FREE tier API

Checking Ollama (local, free):
✓ Ollama is installed and available

Available models:
NAME                ID              SIZE      MODIFIED    
deepseek-r1:1.5b    e0979632db5a    1.1 GB    10 days ago    
llama3.2:latest     a80c4f17acd5    2.0 GB    2 weeks ago    
gemma3:1b           8648f39daa8f    815 MB    2 weeks ago    



In [73]:
# Connect to FREE services

# Groq (free tier API)
if groq_api_key:
    groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
    print("✓ Connected to Groq API (free tier)")
else:
    groq = None
    print("✗ Groq not available (no API key)")

# Ollama (local, free)
ollama_base_url = "http://localhost:11434/v1"
ollama = OpenAI(base_url=ollama_base_url, api_key="ollama")
print("✓ Connected to Ollama at", ollama_base_url)

✓ Connected to Groq API (free tier)
✓ Connected to Ollama at http://localhost:11434/v1


In [74]:
# FREE models

# Groq models (free tier API - very fast!)
GROQ_LLAMA_70B = "llama-3.3-70b-versatile"  # Excellent reasoning
GROQ_LLAMA_8B = "llama-3.1-8b-instant"  # Fast, good for code
GROQ_LLAMA_3_3 = "llama-3.3-70b-versatile"  # Updated: specdec model was decommissioned

# Ollama models (local, free - install with: ollama pull <model_name>)
OLLAMA_QWEN = "qwen2.5-coder:7b"  # Excellent for code generation
OLLAMA_DEEPSEEK = "deepseek-coder-v2:16b"  # Great for code tasks
OLLAMA_CODELLAMA = "codellama:13b"  # Meta's code-focused model
OLLAMA_LLAMA = "llama3.1:8b"  # General purpose, good reasoning

print("FREE models available:")
print("\nGroq (free tier API - very fast):")
print(f"  - {GROQ_LLAMA_70B}")
print(f"  - {GROQ_LLAMA_8B}")
print(f"  - {GROQ_LLAMA_3_3}")
print("\nOllama (local - install with 'ollama pull <model>'):")
print(f"  - {OLLAMA_QWEN}")
print(f"  - {OLLAMA_DEEPSEEK}")
print(f"  - {OLLAMA_CODELLAMA}")
print(f"  - {OLLAMA_LLAMA}")

FREE models available:

Groq (free tier API - very fast):
  - llama-3.3-70b-versatile
  - llama-3.1-8b-instant
  - llama-3.3-70b-versatile

Ollama (local - install with 'ollama pull <model>'):
  - qwen2.5-coder:7b
  - deepseek-coder-v2:16b
  - codellama:13b
  - llama3.1:8b


## PLEASE NOTE:

We will be writing a solution to convert Python into efficient, optimized C++ code for your machine, which can be compiled to native machine code and executed.

It is not necessary for you to execute the code yourself - that's not the point of the exercise!

But if you would like to (because it's satisfying!) then I'm including the steps here. Very optional!

As an alternative, I'll also show you a website where you can run the C++ code.

In [75]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.26200',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'x86_64-w64-mingw32'},
 'package_managers': ['winget'],
 'cpu': {'brand': '11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz',
  'cores_logical': 8,
  'cores_physical': 4,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'gcc.EXE (Rev8, Built by MSYS2 project) 15.2.0',
   'g++': 'g++.EXE (Rev8, Built by MSYS2 project) 15.2.0',
   'clang': '',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [76]:
# Optional: Ask LLM about system setup (skip this if you want - auto-detection below!)

message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please provide recommendations for the best C++ compiler setup for my system.

System information:
{system_info}
"""

# Uncomment below to ask LLM for advice
# if groq:
#     response = groq.chat.completions.create(model=GROQ_LLAMA_70B, messages=[{"role": "user", "content": message}])
# else:
#     response = ollama.chat.completions.create(model=OLLAMA_QWEN, messages=[{"role": "user", "content": message}])
# display(Markdown(response.choices[0].message.content))

print("Skipping LLM query - will auto-detect compiler below...")
    

Skipping LLM query - will auto-detect compiler below...


## C++ Compiler Setup

The cell below will auto-detect if you have a C++ compiler installed (g++, clang++, or MSVC).

**For Windows users:** If no compiler is found, you have these options:

### Option 1: MinGW-w64 via MSYS2 (Recommended - Easiest)

You already have MSYS2 installed! Now install the compiler:

1. **Open MSYS2 UCRT64 terminal** (search for "MSYS2 UCRT64" in Windows Start menu)
   - NOT the plain "MSYS2" terminal - make sure it says **UCRT64**

2. **Update package database:**
   ```bash
   pacman -Syu
   ```
   (Press Enter to confirm, close the terminal if asked, then reopen)

3. **Install gcc/g++ compiler:**
   ```bash
   pacman -S mingw-w64-ucrt-x86_64-gcc
   ```
   (Press Enter to confirm - type Y when asked)

4. **Add to Windows PATH:**
   - Press Windows key, search "Environment Variables"
   - Click "Edit the system environment variables"
   - Click "Environment Variables" button
   - Under "System variables", find and select "Path", click "Edit"
   - Click "New" and add: `C:\msys64\ucrt64\bin`
   - Click OK on all dialogs

5. **Restart VS Code** and rerun the compiler detection cell below

### Option 2: Visual Studio Build Tools
Download from https://visualstudio.microsoft.com/downloads/
- Install "Desktop development with C++"

### Option 3: LLVM/Clang
Download from https://releases.llvm.org/download.html

After installing, restart VS Code and rerun the cell below.

In [77]:
# Auto-detect available C++ compiler on Windows
import shutil

def find_compiler():
    """Find available C++ compiler on the system"""
    # Try to find g++ (MinGW)
    if shutil.which("g++"):
        return "g++", ["g++", "-std=c++17", "-O3", "-march=native", "-DNDEBUG", "main.cpp", "-o", "main.exe"], ["main.exe"]
    
    # Try to find clang++
    if shutil.which("clang++"):
        return "clang++", ["clang++", "-std=c++17", "-O3", "-march=native", "-DNDEBUG", "main.cpp", "-o", "main.exe"], ["main.exe"]
    
    # Try to find MSVC (cl.exe)
    if shutil.which("cl"):
        return "cl", ["cl", "/O2", "/std:c++17", "/EHsc", "main.cpp"], ["main.exe"]
    
    return None, None, None

compiler_name, compile_command, run_command = find_compiler()

if compiler_name:
    print(f"✓ Found {compiler_name} compiler")
    print(f"  Compile command: {' '.join(compile_command)}")
    print(f"  Run command: {' '.join(run_command)}")
else:
    print("✗ No C++ compiler found!")
    print("\n" + "="*60)
    print("INSTALLATION INSTRUCTIONS (you have MSYS2 already!):")
    print("="*60)
    print("\n1. Open 'MSYS2 UCRT64' terminal (NOT plain MSYS2)")
    print("   Search for 'MSYS2 UCRT64' in Windows Start menu")
    print("\n2. Run these commands:")
    print("   pacman -Syu              # Update package database")
    print("   pacman -S mingw-w64-ucrt-x86_64-gcc   # Install gcc")
    print("\n3. Add to Windows PATH:")
    print("   C:\\msys64\\ucrt64\\bin")
    print("\n4. Restart VS Code and rerun this cell")
    print("\n5. OR use online compiler: https://www.programiz.com/cpp-programming/online-compiler/")
    print("="*60)

✓ Found g++ compiler
  Compile command: g++ -std=c++17 -O3 -march=native -DNDEBUG main.cpp -o main.exe
  Run command: main.exe


## And now, on with the main task

In [78]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [79]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [80]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [81]:
def port(client, model, python):
    response = client.chat.completions.create(model=model, messages=messages_for(python))
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [82]:
pi = """
import time

start_time = time.time()
total = sum(range(1000000))
end_time = time.time()

print(f"Sum: {total}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""


In [83]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [84]:
run_python(pi)

Sum: 499999500000
Execution Time: 0.016047 seconds


In [85]:
# Try Groq first (free tier, very fast!)
if groq:
    port(groq, GROQ_LLAMA_70B, pi)
else:
    port(ollama, OLLAMA_QWEN, pi)

# Compiling C++ and executing

The notebook will auto-detect your C++ compiler and set up the right commands.

**Note:** Compiling and running C++ is optional - it's just to see the performance difference!

**Alternative:** You can also test the generated C++ code in an online compiler:
- https://www.programiz.com/cpp-programming/online-compiler/
- https://godbolt.org/ (great for seeing assembly output)
- https://wandbox.org/

Just copy the generated C++ code from `main.cpp` and paste it into the online compiler.

In [86]:
# Compile and run the C++ code

def compile_and_run():
    if compile_command is None:
        print("❌ No C++ compiler available!")
        print("Please install a C++ compiler (see instructions above) or use an online compiler:")
        print("https://www.programiz.com/cpp-programming/online-compiler/")
        return
    
    try:
        # Compile
        result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
        print("✓ Compilation successful!")
        
        # Run 3 times to see timing consistency
        for i in range(3):
            result = subprocess.run(run_command, check=True, text=True, capture_output=True)
            print(result.stdout, end='')
    except subprocess.CalledProcessError as e:
        print(f"❌ Error during compilation/execution:")
        print(e.stderr if e.stderr else str(e))
    except FileNotFoundError:
        print("❌ Compiler not found in PATH!")
        print("Please restart VS Code after installing the compiler.")

In [87]:
compile_and_run()

✓ Compilation successful!
Sum: -363189984
Execution Time: 0.000000 seconds
Sum: -363189984
Execution Time: 0.000000 seconds
Sum: -363189984
Execution Time: 0.000000 seconds


In [88]:
29.066188/0.369109

78.74689590337813

## OK let's try the other FREE models!

In [89]:
# Groq Llama 8B (free, instant)
if groq:
    port(groq, GROQ_LLAMA_8B, pi)
    compile_and_run()
else:
    print("Groq not available - set GROQ_API_KEY to use")

✓ Compilation successful!
Sum: 1783293664
Execution Time: 4.28e-05 seconds
Sum: 1783293664
Execution Time: 4.27e-05 seconds
Sum: 1783293664
Execution Time: 4.29e-05 seconds


In [90]:
# Groq Llama 3.3 70B with speculative decoding (free, very fast!)
if groq:
    port(groq, GROQ_LLAMA_3_3, pi)
    compile_and_run()
else:
    print("Groq not available - set GROQ_API_KEY to use")

✓ Compilation successful!
Sum: 18446744073346361632
Execution Time: 0 seconds
Sum: 18446744073346361632
Execution Time: 0 seconds
Sum: 18446744073346361632
Execution Time: 1e-07 seconds


In [91]:
# Ollama local models - using llama3.2 (already installed)
port(ollama, "llama3.2:latest", pi)
compile_and_run()

❌ Error during compilation/execution:
main.cpp:9:31: error: stray '#' in program
    9 |     #pragma GCC target("avx2")#pragma GCC optimize("Ofast")
      |                               ^
main.cpp:9:32: error: '#pragma GCC target' string is badly formed
    9 |     #pragma GCC target("avx2")#pragma GCC optimize("Ofast")
      |                                ^~~~~~
main.cpp: In function 'int main()':
main.cpp:9:13: error: '#pragma GCC option' is not allowed inside functions
    9 |     #pragma GCC target("avx2")#pragma GCC optimize("Ofast")
      |             ^~~
main.cpp:23:59: error: 'setprecision' is not a member of 'std'
   23 |     std::cout << "Execution Time: " << std::fixed << std::setprecision(6) << elapsed_s << " seconds" << std::endl;
      |                                                           ^~~~~~~~~~~~
main.cpp:4:1: note: 'std::setprecision' is defined in header '<iomanip>'; this is probably fixable by adding '#include <iomanip>'
    3 | #include <chrono>
  +++ |

In [92]:
print(f"""
Using FREE models!

Groq (free tier API - very fast):
- {GROQ_LLAMA_70B}
- {GROQ_LLAMA_8B}
- {GROQ_LLAMA_3_3}

Ollama (local, no API key needed):
- {OLLAMA_QWEN}
- {OLLAMA_DEEPSEEK}
- {OLLAMA_CODELLAMA}
- {OLLAMA_LLAMA}

No costs - experiment freely!
""")


Using FREE models!

Groq (free tier API - very fast):
- llama-3.3-70b-versatile
- llama-3.1-8b-instant
- llama-3.3-70b-versatile

Ollama (local, no API key needed):
- qwen2.5-coder:7b
- deepseek-coder-v2:16b
- codellama:13b
- llama3.1:8b

No costs - experiment freely!

